# 02 - Explore openFDA Drug Labels & Provision Object Storage

**Learning objective:** Understand the openFDA drug label structure and provision your first Scaleway resource with OpenTofu.

We will:
1. Load the bundled openFDA labels dataset
2. Inspect the section structure (boxed_warning, drug_interactions, pregnancy, etc.)
3. Make one live call to the openFDA API
4. **Live-provision** an Object Storage bucket via `tofu apply` (first IaC moment!)
5. Upload the dataset to S3

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()

# Load the bundled dataset
with open("../data/openfda_labels.json") as f:
    labels = json.load(f)

print(f"Loaded {len(labels)} drug labels.")

In [ ]:
# Schema summary: what sections are present across the dataset?
from collections import Counter

section_counts = Counter()
for label in labels:
    for key in label:
        if key != "openfda":
            section_counts[key] += 1

print("Section frequencies across all labels:")
for section, count in section_counts.most_common():
    print(f"  {section}: {count}/{len(labels)}")

In [ ]:
# Sample drug: find warfarin
warfarin = next(
    (l for l in labels if "warfarin" in l.get("openfda", {}).get("generic_name", [""])[0].lower()),
    None,
)

if warfarin:
    print(f"Drug: {warfarin['openfda']['generic_name'][0]}")
    print(f"Brand: {warfarin['openfda']['brand_name'][0]}")
    set_ids = warfarin["openfda"].get("set_id", [])
    print(f"Set ID: {set_ids[0] if set_ids else '(not in bundled dataset)'}")
    print(f"Sections: {[k for k in warfarin if k != 'openfda']}")
else:
    print("Warfarin not found in dataset.")

In [ ]:
# Sample drug_interactions chunk
if warfarin and "drug_interactions" in warfarin:
    di_text = warfarin["drug_interactions"][0]
    print("=== Drug Interactions (first 500 chars) ===")
    print(di_text[:500])

In [ ]:
# Sample boxed_warning chunk
if warfarin and "boxed_warning" in warfarin:
    print("=== Boxed Warning (first 300 chars) ===")
    print(warfarin["boxed_warning"][0][:300])

In [ ]:
# Live openFDA API call - see the raw response shape
import requests

resp = requests.get(
    "https://api.fda.gov/drug/label.json",
    params={"search": "openfda.generic_name:warfarin", "limit": 1},
    timeout=30,
)
raw = resp.json()
print(f"API response keys: {list(raw.keys())}")
print(f"Result sections: {list(raw['results'][0].keys())[:10]}...")
print(f"Total matching labels: {raw['meta']['results']['total']}")

## First IaC moment: Provision an Object Storage bucket

We will now run `tofu apply` to create a Scaleway Object Storage bucket.
This is your first live Infrastructure-as-Code action!

In [ ]:
import subprocess

project_suffix = os.environ["PROJECT_SUFFIX"]
iac_dir = "../iac_snippets/object_storage"

# Write tfvars
tfvars = f"""access_key      = "{os.environ["SCW_ACCESS_KEY"]}"
secret_key      = "{os.environ["SCW_SECRET_KEY"]}"
organization_id = "{os.environ["SCW_DEFAULT_ORGANIZATION_ID"]}"
project_id      = "{os.environ["SCW_DEFAULT_PROJECT_ID"]}"
project_suffix      = "{project_suffix}"
"""
with open(f"{iac_dir}/terraform.tfvars", "w") as f:
    f.write(tfvars)

print("Written terraform.tfvars")

In [ ]:
# Initialize and apply
subprocess.run(["tofu", "init"], cwd=iac_dir, check=True)
result = subprocess.run(
    ["tofu", "apply", "-auto-approve"],
    cwd=iac_dir,
    capture_output=True,
    text=True,
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

In [ ]:
# Get the bucket name from tofu output
result = subprocess.run(
    ["tofu", "output", "-raw", "bucket_name"],
    cwd=iac_dir,
    capture_output=True,
    text=True,
)
bucket_name = result.stdout.strip()
print(f"Bucket created: {bucket_name}")

In [ ]:
# Upload the dataset to S3
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://s3.{os.environ['SCW_DEFAULT_REGION']}.scw.cloud",
    aws_access_key_id=os.environ["SCW_ACCESS_KEY"],
    aws_secret_access_key=os.environ["SCW_SECRET_KEY"],
    region_name=os.environ["SCW_DEFAULT_REGION"],
)

s3.upload_file(
    "../data/openfda_labels.json",
    bucket_name,
    "openfda_labels.json",
)
print(f"Uploaded openfda_labels.json to s3://{bucket_name}/")

## You should now have

- [x] Explored the openFDA label structure
- [x] Seen warfarin's drug_interactions and boxed_warning sections
- [x] Made a live call to the openFDA API
- [x] Provisioned an Object Storage bucket via OpenTofu
- [x] Uploaded the labels dataset to S3

**Next:** `03_provision_pgvector_and_chunk.ipynb`